# BKQuiz — Tách chữ đề thi sau duyệt ảnh đợt 1

Chỉ chạy những ZIP do nút **Xuất ảnh đủ điều kiện sang Colab** tạo ra. Input chỉ gồm ảnh được người dùng chọn hoặc được định tuyến auto_vlm bởi chính sách OCR đã được người dùng cho phép. Ảnh cần duyệt tay, tạm hoãn và bị loại không thuộc input; trang đã có native/OCR text không gửi sang VLM. Sau VLM, dùng **Nhập ZIP kết quả VLM** ở web để duyệt **từng câu hỏi đợt 2**. Notebook không tạo phê duyệt và không xuất SFT tự động.

Nếu đổi lựa chọn ảnh sau lần xuất, hãy xuất input mới và chuyển các input ZIP cũ ra khỏi thư mục Colab inputs trước khi chạy. Kết quả từ ảnh đã bị thu hồi quyết định hoặc không còn khớp chính sách tự động sẽ không được nhập.

Chọn **L4 + High-RAM**, dùng runtime sạch. Bật Omit code cell output when saving. Trọng số và các phiên bản kế thừa từ pipeline CPT; ngân sách OCR toàn trang mới chưa được đo trên GPU.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib,json,os,subprocess,sys,zipfile
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/BKQuiz_SFT_EXAMS')
INPUTS=BASE/'inputs'
RESULTS=BASE/'results'
BASE.mkdir(parents=True,exist_ok=True)
INPUTS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
print('Đặt runtime ZIP ở',BASE,'và các ZIP ảnh đã duyệt ở',INPUTS)


In [ ]:
RUNTIME=BASE/'sft_vlm_runtime_c0901ff84e1c1ace.zip'
EXPECTED_SHA='066e95d468ac5a1508a02ae8d4f826cf4f4f89189d91030c72eccbefa2e688f7'
assert RUNTIME.is_file(), 'Chưa upload runtime ZIP đúng cặp notebook'
assert hashlib.sha256(RUNTIME.read_bytes()).hexdigest()==EXPECTED_SHA, 'Runtime ZIP sai hash'
LOCAL=Path('/content/bkquiz_sft_vlm_runtime')
LOCAL.mkdir(exist_ok=True)
with zipfile.ZipFile(RUNTIME) as z:
    assert set(z.namelist())=={'runtime.py','requirements.txt','runtime.json'}
    m=json.loads(z.read('runtime.json'))
    for name,wanted in m['files_sha256'].items():
        body=z.read(name)
        assert hashlib.sha256(body).hexdigest()==wanted
        (LOCAL/name).write_bytes(body)
    (LOCAL/'runtime.json').write_bytes(z.read('runtime.json'))
assert list(INPUTS.glob('sft_*.zip')), 'Chưa có ảnh được duyệt/xuất; dừng trước khi cài hoặc nạp model'
print('Runtime và input đã có; chưa chạy inference')


In [ ]:
# Venv riêng giữ các package Colab hiện tại không bị ghi đè.
ENV=Path('/content/bkquiz_sft_vlm_env')
PY=ENV/'bin/python'
if not PY.exists(): subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(ENV)],check=True)
subprocess.run([str(PY),'-m','pip','install','torch==2.6.0','torchvision==0.21.0','--index-url','https://download.pytorch.org/whl/cu124/'],check=True)
subprocess.run([str(PY),'-m','pip','install','-r',str(LOCAL/'requirements.txt')],check=True)
subprocess.run([str(PY),'-c',"import torch; assert torch.cuda.is_available(), 'Cần GPU CUDA'; p=torch.cuda.get_device_properties(0); print(p.name,p.total_memory); assert p.total_memory>=20*1024**3, 'Chọn L4 hoặc GPU có >=20GiB VRAM'"],check=True)
subprocess.run(['nvidia-smi'],check=True)


## Chạy và tiếp tục

Mỗi ảnh là một request OCR có deadline; model GPU được giữ qua các batch. Cache chỉ dùng lại khi ảnh, identity, prompt và cấu hình đều khớp. Kết quả lỗi giữ raw output; không tự sửa/giải câu hỏi. Nếu timeout hoặc lỗi nạp model, queue dừng và ghi lỗi rõ ràng. Chạy lại bỏ qua cache đã có; chỉ đặt `RETRY_ERRORS=True` sau khi đã xử lý nguyên nhân. ZIP `_results_...zip` trong results là thứ nhập vào web. Không xem cache/thành công máy là câu đã đạt.

In [ ]:
RETRY_ERRORS=False
MAX_BATCHES=None  # Đặt 1 để thử một gói trước.
command=[str(PY),str(LOCAL/'runtime.py'),'--inputs',str(INPUTS),'--out',str(RESULTS)]
if RETRY_ERRORS: command+=['--retry-errors']
if MAX_BATCHES is not None: command+=['--max-batches',str(MAX_BATCHES)]
env=os.environ.copy(); env['HF_HOME']='/content/bkquiz_sft_models'
subprocess.run(command,check=True,env=env)
print('Kết quả lưu ở',RESULTS,'; tải ZIP kết quả về và nhập lại web duyệt đợt 2')
